[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/JoseGG8/AutoEncoderPi/blob/main/pipeline_autoencoder_cuantico.ipynb)

# 🚀 Pipeline Principal: Autoencoder Cuántico (QAE) para Detección de Fallas en Rodamientos (Dataset Paderborn)
### Paradigma de Detección de Anomalías One-Class (Entrenamiento Exclusivo con Datos Sanos)

---

## 📋 Contexto y Planteamiento del Proyecto
En esta etapa del proyecto, el dataset procesado (`paderborn_features_tratadas.csv`) **está compuesto exclusivamente por registros de motores sanos / normales** (59,867 filas, todas con `Label = 0`).

Este escenario corresponde formalmente al paradigma de **Detección de Anomalías Semi-supervisada / One-Class**:
1. El Autoencoder Cuántico (QAE) se entrena **únicamente** con datos normales para aprender la variedad cuántica (manifold) del comportamiento no defectuoso.
2. Al estar libre de fallas, el espacio latente del autoencoder se optimiza para comprimir y reconstruir la dinámica sana con máxima fidelidad.
3. Posteriormente, cualquier registro que presente una desviación física (fallas en pistas, jaula, bolas o condiciones no vistas) no podrá ser reconstruido eficientemente, produciendo una baja fidelidad en el **SWAP Test** y un **alto error de reconstrucción**.

### 🔄 Flujo Metodológico Completo:
$$\text{DATOS SANOS} \longrightarrow \text{Manejo NaN/Inf} \longrightarrow \log(1+x) \text{ FFT} \longrightarrow \text{Clipping [P1, P99]} \longrightarrow \mathbf{RobustScaler} \longrightarrow \mathbf{Zero-Padding (128)} \longrightarrow \mathbf{Norma } L_2 \longrightarrow \mathbf{Amplitude\ Encoding (7\ Qubits)} \longrightarrow \mathbf{QAE} \longrightarrow \mathbf{SWAP\ Test}$$

> **Nota sobre el estado del dataset:**
> El archivo `paderborn_features_tratadas.csv` en la raíz ya tiene los **Pasos 1 a 3** implementados (verificación de calidad, escala logarítmica en FFT y clipping P1-P99). Este notebook retoma desde el **Paso 4 (RobustScaler)** y estructura la partición, preparación cuántica, entrenamiento y establecimiento de umbrales de detección.


In [ ]:
# ==============================================================================
# CELDA 1: CONFIGURACIÓN DEL ENTORNO E IMPORTACIÓN DE LIBRERÍAS
# ==============================================================================
# Si ejecutas este notebook en Google Colab o un entorno nuevo sin las librerías,
# descomenta la siguiente línea para instalarlas:
# !pip install pennylane scikit-learn matplotlib seaborn pandas numpy

import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

# Herramientas de preprocesamiento clásico y métricas
from sklearn.preprocessing import RobustScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, precision_recall_curve

# Librería de Computación Cuántica (PennyLane)
try:
    import pennylane as qml
    print(f"✅ PennyLane importado correctamente. Versión: {qml.__version__}")
except ImportError:
    print("⚠️ PennyLane no está instalado en este entorno.")
    print("   Para instalarlo ejecuta: pip install pennylane")

# Configuración visual para gráficos
plt.style.use('seaborn-v0_8-whitegrid' if 'seaborn-v0_8-whitegrid' in plt.style.available else 'default')
plt.rcParams['figure.figsize'] = (10, 5)
plt.rcParams['font.size'] = 11

print("✅ Entorno configurado correctamente.")


---
## 🧠 1. Justificación Metodológica: Clipping + RobustScaler en Datos Sanos

Para la sustentación ante el profesor, estas son las razones clave de aplicar ambos métodos:

### 1.1 ¿Qué resuelve el Clipping y qué resuelve RobustScaler?
* **Clipping (Winsorización P1–P99):** Elimina ruidos espurios, picos transitorios y saturaciones de medición que puedan existir incluso en motores considerados sanos. Garantiza que el $98\%$ central de la distribución defina la frontera del espacio operativo.
* **RobustScaler:** Nivela las **unidades físicas dispares** entre las 73 características (ej. RPM $\sim 1500$, Fuerza $\sim 1000\text{ N}$, Kurtosis $\sim 3$, amplitudes FFT $\sim 0.05$).
  $$z = \frac{x - \text{mediana}(X)}{\text{IQR}(X)}$$
  Al usar la mediana y el Rango Intercuartílico ($Q_3 - Q_1$), centra cada variable en 0 y homogeniza su escala sin que los valores extremos residuales alteren la escala.

### 1.2 ¿Por qué utilizarlos consecutivamente?
1. **Clipping antes de RobustScaler:** Asegura que los cuantiles se calculen sobre señales limpias de ruido extremo de medición.
2. **RobustScaler después de Clipping:** Sin él, al calcular la normalización cuántica $L_2$ para Amplitude Encoding, la variable con mayor valor físico (RPM) acapararía toda la amplitud del estado cuántico, dejando las características espectrales de FFT (cruciales para detectar fallas) prácticamente en cero.
3. **Cero Data Leakage:** Todos los parámetros (mediana e IQR) se calculan **únicamente** con el subconjunto de entrenamiento de datos sanos (`X_train_sano`).


In [ ]:
# ==============================================================================
# CELDA 2: CARGA Y VERIFICACIÓN DEL DATASET DE MOTORES SANOS
# ==============================================================================

# Detección flexible de la ruta del archivo (funciona en local y en Colab)
rutas_posibles = [
    "../../paderborn_features_tratadas.csv",                     # 2 niveles arriba (raíz del proyecto)
    "../paderborn_features_tratadas.csv",                        # 1 nivel arriba (algoritmosTratamientoDatos)
    "paderborn_features_tratadas.csv",                           # En la misma carpeta del notebook
    os.path.abspath(os.path.join(os.getcwd(), "..", "..", "paderborn_features_tratadas.csv")),
    "/content/drive/My Drive/Paderborn_QML/paderborn_features_tratadas.csv" # En Google Colab
]

ruta_encontrada = None
for r in rutas_posibles:
    if os.path.exists(r):
        ruta_encontrada = r
        break

if ruta_encontrada is None:
    raise FileNotFoundError("❌ No se encontró 'paderborn_features_tratadas.csv'. Verifica la ruta.")

print(f"📂 Cargando dataset desde: {ruta_encontrada}")
df = pd.read_csv(ruta_encontrada)

print(f"✅ Dimensiones del dataset cargado: {df.shape[0]} filas × {df.shape[1]} columnas")

# Identificación de columnas
col_label = 'Label' if 'Label' in df.columns else [c for c in df.columns if c.lower() in ['label', 'target', 'clase']][0]
feature_cols = [c for c in df.columns if c != col_label]

op_cols = [c for c in ['RPM', 'Torque_Nm', 'Fuerza_N'] if c in df.columns]
stat_cols = [c for c in ['RMS', 'Kurtosis', 'Skewness', 'Crest_Factor'] if c in df.columns]
fft_cols = [c for c in df.columns if c.startswith('fft_')]

print(f"\n📊 Desglose de características identificadas ({len(feature_cols)} en total):")
print(f"   • Condiciones operativas (físicas): {len(op_cols)} columnas -> {op_cols}")
print(f"   • Estadísticas temporales:         {len(stat_cols)} columnas -> {stat_cols}")
print(f"   • Espectrales FFT (con log1p):      {len(fft_cols)} columnas")
print(f"   • Variable objetivo (etiqueta):     '{col_label}'")

# Verificación de integridad y composición de clases
n_nulos = df[feature_cols].isnull().sum().sum()
n_infs = np.isinf(df[feature_cols].values).sum()
print(f"\n🔍 Verificación de calidad:")
print(f"   • Valores nulos detectados:      {n_nulos}")
print(f"   • Valores infinitos detectados:  {n_infs}")
assert n_nulos == 0 and n_infs == 0, "Error: Existen datos corruptos en el dataset tratado."

print(f"\n🏷️ Distribución de etiquetas en el dataset actual:")
conteo_etiquetas = df[col_label].value_counts()
print(conteo_etiquetas.to_string())

if len(conteo_etiquetas) == 1 and 0 in conteo_etiquetas.index:
    print("\nℹ️ Confirmado: El dataset actual contiene 100% registros de motores sanos (Label=0).")
    print("   El pipeline operará bajo el paradigma One-Class Anomaly Detection.")


---
## 🛡️ 2. Partición Estratégica de Datos Sanos (Train / Val / Test)

Dado que en este momento **todos los datos son sanos**, dividimos el dataset en tres conjuntos independientes:

1. **`X_train_sano` (70%):** Utilizado para:
   - Ajustar (`fit`) los parámetros estadísticos del `RobustScaler` (mediana e IQR).
   - Entrenar los parámetros variacionales $\theta$ del Autoencoder Cuántico (QAE).
2. **`X_val_sano` (15%):** Utilizado para:
   - Monitorear la convergencia y fidelidad durante el entrenamiento (prevención de sobreajuste).
   - Ajustar hiperparámetros (learning rate, número de capas del ansätz, batch size).
3. **`X_test_sano` (15%):** Conjunto ciego de prueba de sanos.
   - Utilizado para medir el error de reconstrucción en datos normales no vistos y **calcular el umbral de alarma estadística $\tau$** (línea base de control).

> **Garantía metodológica:** El escalador nunca ve los datos de `val` ni `test` durante su cálculo (`fit`).


In [ ]:
# ==============================================================================
# CELDA 3: PARTICIÓN EN TRAIN (70%), VALIDACIÓN (15%) Y TEST (15%)
# ==============================================================================

X_todos_sanos = df[feature_cols].values

# 1. Separar 70% Train y 30% Temporal (Val + Test)
X_train_sano, X_temp_sano = train_test_split(
    X_todos_sanos, 
    test_size=0.30, 
    random_state=42, 
    shuffle=True
)

# 2. Dividir el 30% temporal en 15% Val y 15% Test
X_val_sano, X_test_sano = train_test_split(
    X_temp_sano, 
    test_size=0.50, 
    random_state=42, 
    shuffle=True
)

print(f"✅ Partición completada exitosamente:")
print(f"   • X_train_sano (Ajuste de Scaler y QAE): {X_train_sano.shape[0]} filas ({X_train_sano.shape[0]/len(df)*100:.1f}%)")
print(f"   • X_val_sano   (Monitoreo de fidelidad): {X_val_sano.shape[0]} filas ({X_val_sano.shape[0]/len(df)*100:.1f}%)")
print(f"   • X_test_sano  (Cálculo de umbral tau):  {X_test_sano.shape[0]} filas ({X_test_sano.shape[0]/len(df)*100:.1f}%)")
print(f"   • Características por registro:          {X_train_sano.shape[1]}")


---
## ⚙️ 3. Paso 4: Ajuste y Transformación con RobustScaler

Ajustamos `RobustScaler` **estrictamente sobre `X_train_sano`** y luego transformamos `train`, `val` y `test`:
- Para cada característica $j$:
  $$z_j = \frac{x_j - \text{mediana}(X_{\text{train}, j})}{\text{IQR}(X_{\text{train}, j})}$$
- **Beneficio directo:** Tras este paso, ninguna variable física domina por su orden de magnitud; todas quedan centradas en 0 con un rango intercuartílico unitario.


In [ ]:
# ==============================================================================
# CELDA 4: AJUSTE, TRANSFORMACIÓN CON ROBUSTSCALER Y EXPORTACIÓN A DISCO
# ==============================================================================

# Instanciar RobustScaler
robust_scaler = RobustScaler()

# 1. FIT EXCLUSIVO EN X_train_sano (evita cualquier fuga de información)
robust_scaler.fit(X_train_sano)

# 2. TRANSFORM EN MEMORIA RAM (para continuar con los pasos cuánticos)
X_train_scaled = robust_scaler.transform(X_train_sano)
X_val_scaled = robust_scaler.transform(X_val_sano)
X_test_scaled = robust_scaler.transform(X_test_sano)

print(f'✅ RobustScaler ajustado en memoria RAM:')
print(f'   • Forma Train escalado: {X_train_scaled.shape}')
print(f'   • Forma Val escalado:   {X_val_scaled.shape}')
print(f'   • Forma Test escalado:  {X_test_scaled.shape}')

# 3. CREAR Y EXPORTAR EL NUEVO DATASET COMPLETO A DISCO (.CSV)
# Transformamos todas las filas del dataset con los parámetros aprendidos en train
X_todos_scaled = robust_scaler.transform(df[feature_cols].values)

# Reconstruir DataFrame con los mismos nombres de columnas y la etiqueta
df_escalado = pd.DataFrame(X_todos_scaled, columns=feature_cols)
df_escalado[col_label] = df[col_label].values

# Ruta de salida (en la raíz del proyecto o en Drive si es Colab)
# Guardar el nuevo dataset escalado en la carpeta actual (PipelineRobustScaler+Clipping)
ruta_csv_salida = 'paderborn_features_con_robust_scaler.csv'

# Guardar el nuevo dataset escalado en la carpeta actual (PipelineRobustScaler+Clipping)
ruta_csv_salida = 'paderborn_features_con_robust_scaler.csv'

df_escalado.to_csv(ruta_csv_salida, index=False)
print(f'\n💾 NUEVO DATASET GUARDADO EN DISCO:')
print(f'   • Archivo:    {ruta_csv_salida}')
print(f'   • Registros:  {df_escalado.shape[0]} filas × {df_escalado.shape[1]} columnas')

# 4. VERIFICACIÓN ESTADÍSTICA
medianas_train = np.median(X_train_scaled, axis=0)
iqr_train = np.percentile(X_train_scaled, 75, axis=0) - np.percentile(X_train_scaled, 25, axis=0)
print(f'\n📊 Verificación de propiedades:')
print(f'   • Mediana promedio en Train (debe ser 0.0): {np.mean(medianas_train):.4f}')
print(f'   • IQR promedio en Train (debe ser 1.0):      {np.mean(iqr_train):.4f}')

# 5. VISUALIZACIÓN COMPARATIVA (ANTES vs DESPUÉS)
cols_ejemplo = op_cols + stat_cols[:2] + fft_cols[:2]
indices_ejemplo = [feature_cols.index(c) for c in cols_ejemplo]

fig, axes = plt.subplots(1, 2, figsize=(14, 4))
axes[0].boxplot([X_train_sano[:, idx] for idx in indices_ejemplo], tick_labels=cols_ejemplo)
axes[0].set_title('Antes de RobustScaler (Escalas físicas dispares)', fontsize=11, fontweight='bold')
axes[0].set_yscale('symlog')
axes[0].tick_params(axis='x', rotation=45)

axes[1].boxplot([X_train_scaled[:, idx] for idx in indices_ejemplo], tick_labels=cols_ejemplo)
axes[1].set_title('Después de RobustScaler (Centrado en 0, IQR homogéneo)', fontsize=11, fontweight='bold')
axes[1].tick_params(axis='x', rotation=45)
axes[1].set_ylim(-3.5, 3.5)

plt.tight_layout()
plt.show()


---
## ⚛️ 4. Paso 5: Zero-Padding al Espacio Cuántico de 7 Cúbits (128 Amplitudes)

### 4.1 La Restricción Cuántica
Para aplicar **Amplitude Encoding**, el espacio de estados de $n$ cúbits tiene dimensión $2^n$.
Con $n = 7$ cúbits:
$$\text{Dimensión} = 2^7 = 128 \text{ amplitudes}$$

Dado que tenemos $D = 71$ características físicas (o 73 según la versión del dataset):
$$\text{Posiciones restantes} = 128 - D \text{ amplitudes vacías (57 o 55 posiciones)}$$

### 4.2 Justificación de los Ceros (Zero-Padding)
1. **Conservación de la energía del vector:** Dado que $0^2 = 0$, la norma euclídea depende estrictamente de las características reales:
   $$\|\mathbf{x}_{\text{padded}}\|_2 = \sqrt{\sum_{i=1}^{D} x_i^2 + \sum_{j=D+1}^{128} 0^2} = \sqrt{\sum_{i=1}^{D} x_i^2}$$
2. **Neutralidad de información:** Asignar cero significa asignar probabilidad $0$ a esos estados de la base. No introduce correlaciones arbitrarias ni distorsiona las relaciones físicas aprendidas en el preprocesamiento.
3. **Especialización del autoencoder:** El circuito aprende que el subespacio correspondiente a las últimas posiciones tiene amplitud nula en condiciones normales.


In [ ]:
# ==============================================================================
# CELDA 5: IMPLEMENTACIÓN DE ZERO-PADDING A 128 DIMENSIONES (7 CÚBITS)
# ==============================================================================

def aplicar_zero_padding(X_matriz, dimension_objetivo=128):
    """
    Extiende una matriz de características (N, D) con ceros a la derecha 
    hasta alcanzar dimension_objetivo (128 amplitudes para 7 cúbits).
    """
    n_samples, n_features = X_matriz.shape
    pad_len = dimension_objetivo - n_features
    
    if pad_len < 0:
        raise ValueError(f"El número de características ({n_features}) supera la dimensión cuántica ({dimension_objetivo}).")
    
    if pad_len == 0:
        return X_matriz.copy()
        
    ceros = np.zeros((n_samples, pad_len), dtype=np.float64)
    X_padded = np.hstack([X_matriz, ceros])
    return X_padded

# Aplicar Zero-Padding a Train, Val y Test
X_train_padded = aplicar_zero_padding(X_train_scaled, dimension_objetivo=128)
X_val_padded = aplicar_zero_padding(X_val_scaled, dimension_objetivo=128)
X_test_padded = aplicar_zero_padding(X_test_scaled, dimension_objetivo=128)

n_ceros = 128 - X_train_scaled.shape[1]
print(f"✅ Zero-Padding completado exitosamente:")
print(f"   • Características físicas originales: {X_train_scaled.shape[1]}")
print(f"   • Amplitudes en cero agregadas:       {n_ceros}")
print(f"   • Dimensión total del vector:         {X_train_padded.shape[1]} (2^7 = 128)")
print(f"   • Tamaño matriz Train padded:         {X_train_padded.shape}")
print(f"   • Tamaño matriz Val padded:           {X_val_padded.shape}")
print(f"   • Tamaño matriz Test padded:          {X_test_padded.shape}")


---
## 📏 5. Paso 6: Normalización $L_2$ vs. Escalamiento Estadístico

### Diferencia conceptual fundamental para la sustentación:
* **Escalamiento Estadístico (`RobustScaler`):** Se aplica **a lo largo de las filas (por columna)** para igualar unidades de medición entre diferentes sensores.
* **Normalización Cuántica ($L_2$):** Se aplica **a lo largo de las columnas (por fila, vector por vector)** para cumplir con el postulado de probabilidad de Born:
  $$|\psi\rangle = \sum_{i=0}^{127} c_i |i\rangle \quad \text{con} \quad \sum_{i=0}^{127} |c_i|^2 = 1 \iff \|\mathbf{c}\|_2 = 1$$

Cada vector $\mathbf{x} \in \mathbb{R}^{128}$ se normaliza como:
$$\mathbf{c} = \frac{\mathbf{x}}{\|\mathbf{x}\|_2} = \frac{\mathbf{x}}{\sqrt{\sum_{i=1}^{128} x_i^2}}$$


In [ ]:
# ==============================================================================
# CELDA 6: NORMALIZACIÓN L2 POR FILA PARA ESTADOS CUÁNTICOS UNITARIOS
# ==============================================================================

def normalizar_l2_cuantico(X_padded):
    """
    Normaliza cada vector de 128 dimensiones fila por fila para que tenga norma L2 = 1.0,
    garantizando un estado cuántico físico válido para Amplitude Encoding.
    """
    normas = np.linalg.norm(X_padded, axis=1, keepdims=True)
    normas_seguras = np.where(normas == 0, 1.0, normas)
    return X_padded / normas_seguras

# Generar estados cuánticos normalizados
X_train_q = normalizar_l2_cuantico(X_train_padded)
X_val_q = normalizar_l2_cuantico(X_val_padded)
X_test_q = normalizar_l2_cuantico(X_test_padded)

# Verificación de normas unitarias
normas_train = np.linalg.norm(X_train_q, axis=1)
normas_test = np.linalg.norm(X_test_q, axis=1)

print(f"✅ Normalización L2 completada:")
print(f"   • Norma mínima en Train: {np.min(normas_train):.6f}")
print(f"   • Norma máxima en Train: {np.max(normas_train):.6f}")
print(f"   • Suma de amplitudes al cuadrado (|c_i|^2) registro 0: {np.sum(X_train_q[0]**2):.6f}")
print(f"   • ¿Todos los estados cumplen ||c||_2 = 1?: {np.allclose(normas_train, 1.0) and np.allclose(normas_test, 1.0)}")


---
## 🔬 6. Pasos 7 y 8: Arquitectura del Autoencoder Cuántico (QAE) y SWAP Test

### 6.1 Estructura del Circuito
El QAE comprime un estado de $N = 7$ cúbits a un cuello de botella de $K$ cúbits latentes ($K < 7$):
- **Cúbits de Entrada:** 7 cúbits ($2^7 = 128$ amplitudes).
- **Cúbits Latentes ($K$):** Contienen la representación comprimida de la señal.
- **Trash Qubits ($M = 7 - K$):** Cúbits descartados durante la compresión.

```
|ψ_in⟩ ───[ 7 cúbits ]─── [ Ansätz Variacional U(θ) ] ──┬── [ K Latentes ] (Cuello de botella)
                                                        │
                                                        └── [ M Trash Qubits ] ─── Proyección / SWAP Test
```

### 6.2 ⚠️ ¿Qué significa una fidelidad inicial del 25%? (Línea Base Cuántica)
En el Autoencoder Cuántico, medimos la probabilidad de que los **trash qubits** colapsen en el estado fundamental $|0\dots0angle$.

Es crucial comprender de dónde proviene matemáticamente la fidelidad de partida:
- Si usamos $M = 2$ trash qubits ($K=5$ latentes), el espacio de los trash qubits tiene $2^M = 2^2 = 4$ estados de base computacional:
  $$|00angle, \quad |01angle, \quad |10angle, \quad |11angle$$
- Antes de entrenar (con pesos aleatorios no convergidos), el estado cuántico se distribuye de forma aproximadamente equiprobable entre los 4 estados. Por lo tanto:
  $$	ext{Fidelidad Base} = P(|00angle) = rac{1}{2^M} = rac{1}{4} = \mathbf{25.0\%}$$
- **Conclusión fundamental:** **¡Un 25% NO significa que el modelo no sirva!** Es la **línea base inicial aleatoria (Random Baseline)** de un sistema con 2 trash qubits.
  - Con $M = 1$ trash qubit ($K=6$), la línea base inicial es $1/2 = \mathbf{50.0\%}$.
  - Con $M = 2$ trash qubits ($K=5$), la línea base inicial es $1/4 = \mathbf{25.0\%}$.
  - Con $M = 3$ trash qubits ($K=4$), la línea base inicial es $1/8 = \mathbf{12.5\%}$.
- El objetivo del entrenamiento es **hacer subir esa probabilidad desde el 25% (o 50%) hacia valores altos (70%, 80%, 90%+)** mediante el descenso de gradiente.


In [ ]:
# ==============================================================================
# CELDA 7: CIRCUITO QAE Y SWAP TEST IMPLEMENTADO EN PENNYLANE
# ==============================================================================
from pennylane import numpy as pnp

TOTAL_QUBITS = 7      # 2^7 = 128 amplitudes

# Configuración del cuello de botella:
# Recomendado para Fase 1: K = 6 (1 trash qubit, baseline inicial = 50%)
# Para compresión más alta: K = 5 (2 trash qubits, baseline inicial = 25%)
LATENT_QUBITS = 6     # K = 6 cúbits latentes (1 trash qubit)
TRASH_QUBITS = TOTAL_QUBITS - LATENT_QUBITS

# Dispositivo cuántico
dev = qml.device("default.qubit", wires=TOTAL_QUBITS)

def ansatz_qae(weights, wires):
    """
    Ansätz variacional con rotaciones parametrizadas RY y RZ
    y entrelazamiento circular CNOT.
    weights shape: (n_layers, n_wires, 2)
    """
    n_wires = len(wires)
    for layer in weights:
        for i, wire in enumerate(wires):
            qml.RY(layer[i, 0], wires=wire)
            qml.RZ(layer[i, 1], wires=wire)
            
        for i in range(n_wires - 1):
            qml.CNOT(wires=[wires[i], wires[i + 1]])
        if n_wires > 2:
            qml.CNOT(wires=[wires[-1], wires[0]])

trash_wires = list(range(LATENT_QUBITS, TOTAL_QUBITS))

@qml.qnode(dev)
def qae_qnode(state_vector, weights):
    # 1. Amplitude Embedding
    qml.AmplitudeEmbedding(features=state_vector, wires=range(TOTAL_QUBITS), normalize=False)
    
    # 2. Ansätz de compresión
    ansatz_qae(weights, wires=range(TOTAL_QUBITS))
    
    # 3. Medición proyectiva (SWAP Test): probabilidad de medir |0...0> en trash_wires
    return qml.probs(wires=trash_wires)

def calcular_fidelidad(state_vector, weights):
    """Devuelve la fidelidad F = P(|0...0> en trash qubits)."""
    probs = qae_qnode(state_vector, weights)
    return probs[0]  # Índice 0 corresponde al estado |00...0>

print(f"✅ Circuito cuántico QAE configurado en PennyLane:")
print(f"   • Cúbits totales:         {TOTAL_QUBITS} (Espacio 2^{TOTAL_QUBITS} = 128)")
print(f"   • Cúbits latentes (K):    {LATENT_QUBITS}")
print(f"   • Trash qubits:           {TRASH_QUBITS} (wires {trash_wires})")
print(f"   • Línea base inicial:     {100.0 / (2**TRASH_QUBITS):.1f}% (probabilidad aleatoria sin entrenar)")


---
## 📈 7. Estrategia de Escalamiento Progresivo por Número de Registros

Atendiendo al requerimiento del profesor:
- **Las 73 características se mantienen fijas en todo momento.**
- El escalamiento del dataset se realiza **aumentando progresivamente el número de registros (filas) de entrenamiento**:

| Fase | Objetivo Principal | Registros Sugeridos | Propósito Técnico |
| :--- | :--- | :--- | :--- |
| **Fase 1: Prototipo y Depuración** | Verificar convergencia, ausencia de Barren Plateaus y calibrar learning rate. | $100 - 250$ filas sanas | Simulación cuántica rápida en CPU/GPU sin esperas prolongadas. |
| **Fase 2: Estabilización Intermedia** | Evaluar generalización en `X_val_sano` y fijar número de capas del ansätz. | $1,000 - 2,500$ filas sanas | Balance óptimo entre representatividad de modos de vibración y tiempo. |
| **Fase 3: Escalamiento Mayor** | Modelo definitivo de alta robustez. | $5,000+$ filas sanas (hasta el total) | Optimización por mini-batches para aprender toda la envolvente sana. |

> **Nota:** Las cantidades son ejemplos adaptables a la capacidad de cómputo disponible.


In [ ]:
# ==============================================================================
# CELDA 8: RUTINA DE ENTRENAMIENTO PROGRESIVO CON GRADIENTES ACTIVOS
# ==============================================================================

# 1. Inicialización de pesos con el submódulo PennyLane NumPy (requerido para Autograd)
n_layers = 2
np.random.seed(42)

# IMPORTANTE: requires_grad=True activa el rastreo de derivadas en PennyLane
initial_weights = pnp.random.uniform(
    low=0.0, 
    high=2.0 * np.pi, 
    size=(n_layers, TOTAL_QUBITS, 2), 
    requires_grad=True
)

def loss_batch(weights_var, batch_data):
    """
    Función de costo diferenciable para PennyLane:
    Minimiza el error de reconstrucción medio: 1 - Fidelidad media.
    """
    costo_acumulado = 0.0
    for sample in batch_data:
        probs = qae_qnode(sample, weights_var)
        fidelidad = probs[0]  # P(|0...0>)
        costo_acumulado = costo_acumulado + (1.0 - fidelidad)
    return costo_acumulado / len(batch_data)

# Seleccionar muestra para Fase 1 (Prototipo rápido)
N_SAMPLES_FASE1 = 100
X_demo = X_train_q[:N_SAMPLES_FASE1]

# Medir fidelidad previa al entrenamiento (Línea Base)
fidelidades_iniciales = [float(calcular_fidelidad(s, initial_weights)) for s in X_demo[:20]]
print(f"📊 Fidelidad inicial antes de optimizar: {np.mean(fidelidades_iniciales) * 100:.2f}%")
print(f"🚀 Iniciando entrenamiento progresivo - FASE 1 ({N_SAMPLES_FASE1} registros sanos)...")

# Optimizador Adam de PennyLane con tasa de aprendizaje ajustada
opt = qml.AdamOptimizer(stepsize=0.12)
weights = initial_weights.copy()

history_loss = []
history_fidelity = []

n_epochs = 15
batch_size = 25

for epoch in range(n_epochs):
    perm = np.random.permutation(len(X_demo))
    epoch_losses = []
    
    for i in range(0, len(X_demo), batch_size):
        batch = X_demo[perm[i:i + batch_size]]
        # Actualización de pesos mediante Autograd
        weights, cost = opt.step_and_cost(lambda w: loss_batch(w, batch), weights)
        epoch_losses.append(float(cost))
        
    avg_loss = np.mean(epoch_losses)
    avg_fidelity = 1.0 - avg_loss
    history_loss.append(avg_loss)
    history_fidelity.append(avg_fidelity)
    
    print(f"   Época {epoch + 1:02d}/{n_epochs:02d} | Pérdida (1 - F): {avg_loss:.4f} | Fidelidad Media: {avg_fidelity * 100:.2f}%")

# Visualizar la curva de aprendizaje (demostrando la subida respecto a la línea base)
fig, ax1 = plt.subplots(figsize=(9, 4))
ax1.plot(history_loss, 'o-', color='#e74c3c', label='Pérdida (1 - Fidelidad)')
ax1.set_xlabel('Época')
ax1.set_ylabel('Pérdida de Reconstrucción', color='#e74c3c')
ax1.set_title(f'Convergencia del QAE (Fase 1: K={LATENT_QUBITS} Latentes, {TRASH_QUBITS} Trash Qubits)', fontweight='bold')

ax2 = ax1.twinx()
ax2.plot([f * 100 for f in history_fidelity], 's--', color='#2ecc71', label='Fidelidad (%)')
ax2.axhline(100.0 / (2**TRASH_QUBITS), color='gray', linestyle=':', label='Línea Base Aleatoria')
ax2.set_ylabel('Fidelidad (%)', color='#2ecc71')

plt.tight_layout()
plt.show()


---
## 🗜️ 8. Experimentación con la Tasa de Compresión del Cuello de Botella

La tasa de compresión en el QAE viene dada por el número de cúbits latentes $K$:
$$\text{Tasa de Compresión} = 1 - \frac{K}{N} = 1 - \frac{K}{7}$$

* **$K = 6$ (1 Trash Qubit):** Compresión suave (14.3%).
* **$K = 5$ (2 Trash Qubits):** Compresión intermedia (28.6%). Punto de balance recomendado.
* **$K = 4$ (3 Trash Qubits):** Compresión agresiva (42.9%). Mayor exigencia representacional.


In [ ]:
# ==============================================================================
# CELDA 9: CONFIGURACIONES EXPERIMENTALES DE COMPRESIÓN
# ==============================================================================

configs_compresion = [
    {"K": 6, "trash": 1, "tasa": "14.3%"},
    {"K": 5, "trash": 2, "tasa": "28.6%"},
    {"K": 4, "trash": 3, "tasa": "42.9%"}
]

print("📊 Opciones de cuello de botella para experimentación:")
for c in configs_compresion:
    print(f"   • K = {c['K']} latentes | {c['trash']} trash qubits | Tasa de Compresión: {c['tasa']}")


---
## 🎯 9. Protocolo de Evaluación y Control Estadístico en Solo Sanos

### 9.1 ¿Cómo se establece el umbral de alarma cuando solo tenemos datos sanos?
En control estadístico de procesos y detección no supervisada, el umbral de alarma $\tau$ se define a partir de la distribución de error en el conjunto de prueba de sanos (`X_test_q`):

1. **Percentil de Control:**
   $$\tau = \text{Percentil}_{95}(E_{\text{sano}}) \quad \text{o} \quad \tau = \text{Percentil}_{99}(E_{\text{sano}})$$
   Esto garantiza de forma teórica que la tasa admisible de falsas alarmas (FPR) esté acotada al $5\%$ o $1\%$.
2. **Criterio de Alarma:**
   - Si $E(\mathbf{x}) \le \tau \implies \mathbf{x}$ se clasifica como **Condición Normal (Sano)**.
   - Si $E(\mathbf{x}) > \tau \implies \mathbf{x}$ se clasifica como **Anomalía / Potencial Falla**.

### 9.2 Validación de Sensibilidad ante Perturbaciones
Para verificar que el circuito cuántico entrenado reacciona ante anomalías antes de recibir el dataset de fallas físicas del laboratorio:
- Evaluamos `X_test_q` (sanos no vistos).
- Evaluamos una muestra con **perturbación espectral sintética** (simulando armónicos de falla en rodamientos).
- Comprobamos el contraste en el error del SWAP Test y dejamos preparada la función `evaluar_nuevas_fallas()` para procesar datos de fallas reales en cuanto estén disponibles.


In [ ]:
# ==============================================================================
# CELDA 10: EVALUACIÓN DE SANOS, DETERMINACIÓN DEL UMBRAL Y SIMULACIÓN DE FALLA
# ==============================================================================

# 1. Evaluar error de reconstrucción en datos sanos de prueba (X_test_q)
N_EVAL = min(100, len(X_test_q))
errores_test_sano = [1.0 - calcular_fidelidad(x, weights) for x in X_test_q[:N_EVAL]]

# 2. Establecer el umbral de alarma estadística tau (percentil 95 -> FPR <= 5%)
tau_p95 = np.percentile(errores_test_sano, 95)
tau_p99 = np.percentile(errores_test_sano, 99)

print(f"📊 LÍNEA BASE DE CONTROL ESTADÍSTICO (DATOS SANOS TEST):")
print(f"   • Error medio de reconstrucción en sanos:   {np.mean(errores_test_sano):.4f}")
print(f"   • Desviación estándar del error:           {np.std(errores_test_sano):.4f}")
print(f"   • Umbral de Alarma tau (Percentil 95%):     {tau_p95:.4f}")
print(f"   • Umbral de Alarma tau (Percentil 99%):     {tau_p99:.4f}")

# 3. Prueba de Sensibilidad: Simular anomalía espectral de falla mecánica
# (Para verificar que el SWAP Test detecta desviaciones antes de incorporar fallas reales)
X_anomalia_simulada = X_test_q[:N_EVAL].copy()
# Inyectar alteración en componentes espectrales (simula armónicos de falla de pista interna)
X_anomalia_simulada[:, 10:25] += np.random.normal(loc=0.4, scale=0.1, size=X_anomalia_simulada[:, 10:25].shape)
X_anomalia_simulada = normalizar_l2_cuantico(X_anomalia_simulada)

errores_anomalia = [1.0 - calcular_fidelidad(x, weights) for x in X_anomalia_simulada]

# Tasa de detección sobre la muestra anómala usando el umbral de sanos
detecciones = np.sum(np.array(errores_anomalia) > tau_p95)
recall_anomalia = (detecciones / len(errores_anomalia)) * 100

print(f"\n🧪 PRUEBA DE SENSIBILIDAD ANTE PERTURBACIÓN:")
print(f"   • Error medio en muestra con anomalía:      {np.mean(errores_anomalia):.4f}")
print(f"   • Detección de anomalías con tau_p95:       {recall_anomalia:.1f}%")

# Gráfica de distribución de errores y umbral de control
plt.figure(figsize=(9, 4))
plt.hist(errores_test_sano, bins=15, alpha=0.7, color='#2ecc71', label='Datos Sanos de Prueba (Normal)')
plt.hist(errores_anomalia, bins=15, alpha=0.7, color='#e74c3c', label='Muestra con Anomalía/Falla')
plt.axvline(tau_p95, color='black', linestyle='--', linewidth=2, label=f'Umbral \tau P95 ({tau_p95:.3f})')
plt.xlabel('Error de Reconstrucción (1 - Fidelidad)')
plt.ylabel('Frecuencia')
plt.title('Control Estadístico y Detección de Anomalías mediante SWAP Test', fontweight='bold')
plt.legend()
plt.tight_layout()
plt.show()

# ==============================================================================
# FUNCIÓN DE UTILIDAD: LISTA PARA PROCESAR NUEVOS REGISTROS DE FALLA CUANDO LLEGUEN
# ==============================================================================
def evaluar_nuevas_fallas(df_nuevas, scaler, weights, tau_decision):
    """
    Procesa un DataFrame nuevo que contenga registros de falla reales:
    Aplica RobustScaler (con los parámetros de sanos) -> Padding -> Norma L2 -> SWAP Test.
    """
    X_raw = df_nuevas[feature_cols].values
    X_scaled = scaler.transform(X_raw)
    X_pad = aplicar_zero_padding(X_scaled, 128)
    X_q = normalizar_l2_cuantico(X_pad)
    
    errores = [1.0 - calcular_fidelidad(x, weights) for x in X_q]
    tasa_alarma = np.mean(np.array(errores) > tau_decision) * 100
    print(f"✅ Evaluación de {len(df_nuevas)} registros completada.")
    print(f"   • Error medio: {np.mean(errores):.4f}")
    print(f"   • Porcentaje clasificado como falla (> tau): {tasa_alarma:.2f}%")
    return errores


---
## 🎓 10. Conclusiones y Resumen para Sustentación del Proyecto

Resumen de defensa ante el profesor adaptado al dataset actual de **motores sanos**:

1. **Paradigma One-Class:**
   El entrenamiento con 100% registros sanos es el estándar en monitoreo de salud estructural (SHM) y mantenimiento predictivo, ya que en la industria real las fallas son raras o no etiquetadas previamente.
2. **Clipping + RobustScaler:**
   El clipping protegió la frontera sana de picos transitorios; `RobustScaler` eliminó la disparidad de magnitudes físicas sin ser alterado por dispersiones residuales.
3. **División Estricta (Train / Val / Test):**
   A pesar de tener solo sanos, se realizó una partición $70/15/15$. El `RobustScaler` solo aprendió de `Train`, y `Test` se reservó para calcular la línea base de normalidad.
4. **Espacio Cuántico (128 Amplitudes en 7 Cúbits):**
   Las características físicas se preservan en las primeras posiciones y el resto se completa con **ceros estructurados**, garantizando un estado físico unitario y sin correlaciones artificiales.
5. **Umbral Estadístico $\tau$:**
   Se establece mediante percentiles ($P95$ o $P99$) sobre los propios datos sanos, fijando rigurosamente la tasa admisible de falsas alarmas antes de contrastar contra registros de falla.
